In [ ]:
import os
import SimpleITK as sitk
import numpy as np
import glob

axial_harmo1 = r'/path/to/workspace/classificationmodel_harmonizednew/axial_harmonize/train/'
axial_harmo2 = r'/path/to/workspace/classificationmodel_harmonizednew/axial_harmonize/val/'
image_path1 = glob.glob(os.path.join(axial_harmo1, '*.nii.gz'))
image_path2 = glob.glob(os.path.join(axial_harmo2, '*.nii.gz'))
image_path = image_path1 + image_path2
print(image_path)
print(len(image_path))

In [ ]:
import os
id = os.path.basename(image_path[1]).split('_')[1]
print(id)

In [ ]:
def calculate_center(patch):
    """
    Calculate the center point of a patch.
    :param patch: SimpleITK Image object representing the patch.
    :return: (x, y, z) coordinates of the center point.
    """
    array = sitk.GetArrayFromImage(patch) # from (z, y, x)
    non_zero_coords = np.argwhere(array > 0)  # Find non-zero pixels
    center = np.mean(non_zero_coords, axis=0)  # Calculate center point
    return tuple(center.astype(int)[::-1])  # from (z, y, x) to (x, y, z)


In [ ]:
def center_crop1(image, center, crop_size):
    """
    Perform center cropping on the original image based on the center point.
    :param image: SimpleITK Image object.
    :param center: (x, y, z) coordinates of the center point.
    :param crop_size: (width, height, depth) cropping size.
    :return: Cropped SimpleITK Image.
    """
    # Get image size (x, y, z)
    size = image.GetSize()
    spacing = image.GetSpacing()

    # Calculate start position for cropping
    start = [max(0, min(int(center[i]) - int(crop_size[i]) // 2, int(size[i]) - int(crop_size[i]))) for i in range(3)]

    # Adjust cropping size if it exceeds image dimensions
    adjusted_size = [min(int(crop_size[i]), int(size[i]) - int(start[i])) for i in range(3)]

    # Use SimpleITK RegionOfInterest to crop the image
    cropped_image = sitk.RegionOfInterest(image, adjusted_size, start)

    # Adjust spatial information for the cropped image
    new_origin = [image.GetOrigin()[i] + start[i] * spacing[i] for i in range(3)]
    cropped_image.SetOrigin(new_origin)
    cropped_image.SetSpacing(spacing)
    cropped_image.SetDirection(image.GetDirection())

    return cropped_image


In [ ]:
import SimpleITK as sitk
import numpy as np
import os

new_dir = r'/path/to/workspace/classificationmodel_harmonizednew/axial_harmonize/rectum_centrecrop2/train_val'
crop_size = (192, 192, 36)

patch_dir = r"/path/to/workspace/classification_model_originalimage/totalsegment_work/resampled/train_val_patches/all_patches"

for img_path in image_path:
    img = sitk.ReadImage(img_path)
    patient_id = os.path.basename(img_path).split('_')[1]

    search_pattern = os.path.join(patch_dir, f"{patient_id}*.nii.gz")
    matching_patches = glob.glob(search_pattern)

    if matching_patches:
        print(f"found patches for patient {patient_id}:")
        for patch in matching_patches:
            print(patch)
            patch_img = sitk.ReadImage(patch)

            # Calculate the center point of the patch
            center_point = calculate_center(patch_img)
            print(f"Center point of the patch: {center_point}")

            # Perform center cropping on the original image
            cropped_img = center_crop1(img, center_point, crop_size)

            # Save the cropped image
            file_name = patient_id + '.nii.gz' #patient number
            output_path = os.path.join(new_dir, file_name)

            sitk.WriteImage(cropped_img, output_path)
            print(f"Cropped image saved to {output_path}")
            
    else:
        print(f"no patches found for patient {patient_id}")
            
print('done')